# La venta que cambió de puesto

Una tienda quiere destacar sus productos estrella. ¿Lo que más factura sigue siendo lo que más deja al registrar créditos?

**45 minutos · Delegar, verificar y firmar.** El entorno y los datos se preparan antes de clase. Si se reinicia la sesión, ejecuta desde la primera celda. No hay claves ni llamadas a modelos en el cuaderno.

Rutas disponibles: D31-A · neto por producto; D31-E · ranking bruto/neto; D31-B · líneas frente a facturas sin ID.

**Antes de ejecutar:** registra una predicción y vota. Las salidas se revelan después de tu elección.

In [ ]:
from pathlib import Path
import json, importlib.util

# Cambia únicamente estas dos frases con tu grupo.
PREGUNTA = 'D31-A'
PREDICCION = ''

# El docente prepara la carpeta antes de clase; no descargamos ni instalamos aquí.
roots = [Path.cwd(), *Path.cwd().parents]
LAB_ROOT = next((p for r in roots for p in (r/'discovery_labs', r) if (p/'catalog/questions.json').exists() and (p/'scripts/run_pilots.py').exists()), None)
if LAB_ROOT is None:
    raise RuntimeError('No encuentro el paquete offline. Pide al docente la carpeta preparada antes de continuar.')
bank = json.loads((LAB_ROOT/'catalog/questions.json').read_text())
available = {q['id']: q for q in bank if q['source_id'] == 'uci-retail' and q['status'] == 'TECHNICALLY_EXECUTED'}
if PREGUNTA not in available:
    raise ValueError('Elige una de las rutas habilitadas de este piloto.')
q = available[PREGUNTA]
print('Pregunta:', q['question'])
print('Métrica:', q['metric'])
print('Denominador:', q['denominator'])

## Dos caminos, una elección humana

Camino 1: sumar importes firmados por producto mediante groupby. Camino 2: reconstruir bruto menos créditos con un acumulador Decimal independiente. Sensibilidad: repetir sin duplicados y contar faltantes por línea/factura.

Pide a tu copiloto que critique ambos caminos. Escribe en tu registro cuál eliges, qué columna podría faltar y qué observación refutaría tu predicción. Puedes cambiar la pregunta si no puede responderse con estos datos.

Verifica dos filas y comprueba un caso a mano antes de mirar la salida.

In [ ]:
if not PREDICCION.strip():
    raise ValueError('Registra primero tu predicción en la celda inicial y vuelve a ejecutarla.')
spec = importlib.util.spec_from_file_location('lab_runner', LAB_ROOT/'scripts/run_pilots.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
data, manifest = lab.load('uci-retail')
print('Filas verificadas:', len(data))
print('Columnas:', ', '.join(data.columns))
print('Hash coincide:', manifest['snapshot_sha256'] == lab.sha(LAB_ROOT/'pilots'/'uci-retail'/'data/snapshot.csv'))
from IPython.display import display
display(data.head(4))

## Ejecutar y cuestionar

La ejecución calcula también sensibilidades predefinidas. Usa sólo la evidencia pertinente a la pregunta votada. No cambies de hipótesis después de ver un resultado sin registrarlo como exploración nueva.

In [ ]:
result = lab.run('uci-retail', LAB_ROOT/'output')
for name in ('invoices', 'eligible_rows', 'unique_games', 'home_win_rate', 'brier', 'host_episodes', 'tree_counterexample_count', 'jaccard_euclidean_ARI'):
    if name in result: print(name, ':', result[name])
from IPython.display import Image
chart = ('denominators.png' if PREGUNTA == 'D31-B' else 'phases.png' if PREGUNTA == 'D10-D2' else 'beliefs.png' if PREGUNTA in ('D85-D1','D85-D2','D85-B') else 'comparison.png')
display(Image(filename=str(LAB_ROOT/'output'/'uci-retail'/chart)))
if PREGUNTA == 'D10-D2': display(lab.pd.DataFrame(result['phases']))
if PREGUNTA == 'D85-D1': display(lab.pd.read_csv(LAB_ROOT/'output'/'uci-retail'/'counterexamples.csv').head())
if PREGUNTA == 'D85-B': display(lab.pd.DataFrame([{'grupo':'Anfitrión','episodios':result['host_episodes'],'fracción con árboles':result['host_tree_fraction']},{'grupo':'Invitados','episodios':result['guest_episodes'],'fracción con árboles':result['guest_tree_fraction']}]))
print('Prueba en contra:', result['adversarial_result'])
print('Límites:', result['inference'])

## Verificar y firmar

1. Comprueba dos filas, un cálculo a mano y los totales de entrada/salida.
2. Abre la tabla CSV generada y contrasta la prueba adversarial.
3. Registra qué le corregiste a la IA y si tu predicción se sostiene.
4. Firma una afirmación con magnitud, denominador, límite y decisión.

Si no alcanza el dato, explica exactamente qué falta. No compartas nombres ni respuestas personales en el repositorio público. Esta actividad es formativa y no modifica tus calificaciones.